# Predicting Galaxy Redshift with Decision Trees and Ensemble Methods 

This notebook was inspired by [Predicting galaxy redshift via regression on 3D-HST photometry](https://spacetelescope.github.io/hellouniverse/notebooks/hello-universe/Regressing_3D-HST_galaxy_redshift_with_decision_trees/Regressing_3D-HST_galaxy_redshift_with_decision_trees.html).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn import tree
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import fetch_california_housing
import joblib
import pandas as pd
from sklearn.model_selection import GridSearchCV, KFold
import plotly.colors as colors
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import tarfile
from astropy.utils.data import download_file
from astropy.table import Table
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

%matplotlib inline

## Download and Examine Dataset

Let's get started by downloading out data from astropy and converting it into a pandas dataframe.

In [ ]:
# First, download the 3D-HST catalog from the MAST archive. This dataset is described in Skelton et. al 2014.
file_url = 'https://archive.stsci.edu/missions/hlsp/3d-hst/RELEASE_V4.0/Photometry/3dhst_master.phot.v4.1.tar'
tarfile.open(download_file(file_url, cache=True), "r:").extract('3dhst_master.phot.v4.1/3dhst_master.phot.v4.1.cat', '.')
# Read the combined photmetric catalog into a dataframe via astropy
tab = Table.read('3dhst_master.phot.v4.1/3dhst_master.phot.v4.1.cat', format='ascii').to_pandas()

The data we use for the tutrial is from the [3D-HST survey](https://archive.stsci.edu/prepds/3d-hst/).The dataset contains Hubble Space Telescope (HST) data.The dataset contains standard information such as target id, field name, coordinates, fluxes, errors, and various photometric flags. In addition, there are derived properties such as photometric redshift (z_peak), spectroscopic redshift (z_spec), mass (lmass) and dust extinction in the V band (Av). Let's examine the contents.

In [ ]:
tab

In [ ]:
tab.columns

### 0.1 Perform Data Cleaning Steps

We will use the same code to clean and transform our data as we did in the linear regression notebooks yesterday.  The comments in the code below highlights the decisions we made.

In [ ]:
# filter data points based on lmass values with full coverage of z_spec
tab = tab[tab.lmass > 9].copy()

# define target
target = 'z_spec'

# Next, we will remove sources which have no constraints for our target variable:
tab = tab[(tab[target] > 0)] 

# Next, we will encode the ‘field” variable, using one hot encoder
tab = pd.get_dummies(tab, columns=['field'],dtype=float)

# Create list of features wie will use for our model
features = [col for col in tab.columns if (col != target)]

# Remove unneeded features 
# The ‘Av’, ‘lmass’ and ‘z_peak’ values were all computed via FAST photometric fit, and so we will exclude them as well. 
# In addition, we will exclude the categorical flag variables (‘flags’, ‘f140w_flag’, ‘star_flag’, ‘use_phot’, ‘near_star’).
# Finally, we will remove field and field_UDS as they are needed after pd.get_dummies
features = [col for col in features if (col != 'Av') and (col != 'lmass') and (col != 'z_peak') 
            and (col != 'flags') and (col != 'f140w_flag') and (col != 'star_flag') 
            and (col != 'use_phot') and (col != 'near_star') and (col != 'field') 
            and (col != 'field_UDS')]

# Finally we will impute missing values of photometric errors (set to -99. in the table) by assigning them the median of the distribution:
errors = [col for col in features if (col[:1] == 'e') and (col[-1:] == 'W')]
for error in errors:
    tab[error] = np.where(tab[error] < -90, tab[error].median(), tab[error])

### 0.2 Divide the Data into Train, Test and Validation sets

In [ ]:
# Divide the data into train, test and validation sets
X = tab[features].values
y = tab[target].values
indices = np.arange(len(y))

# first reserve 70% of the data for training, 30% for validation
X_train, X_validate, y_train, y_validate, indices_train, indices_validate = train_test_split(X, y, indices, test_size=0.3, random_state=42)

# second, split the validation set in half to obtain validation and test sets. 
X_validate, X_test, y_validate, y_test, indices_validate, indices_test = train_test_split(X_validate, y_validate, indices_validate, test_size=0.5, random_state=42)

### DecisionTreeRegressor

Decision trees can also be applied to regression problems, using the DecisionTreeRegressor class.

As in the classification setting, the fit method will take as argument arrays X and y, only that in this case y is expected to have floating point values instead of integer values.  Let's see what performance we can get with default hyperpameters values.

In [ ]:
# Initialize the model
dtree = DecisionTreeRegressor()

# Fit the model parameters using the training dataset
dtree.fit(X_train, y_train)

y_predict = dtree.predict(X_test)
print(f'MSE of Decision Tree model with default hyperparameters = {mean_squared_error(y_test, y_predict):.4f}')

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.scatter(y_test, y_predict, alpha=0.2, color='black')
ax.set_aspect('equal')
ax.set_xlim(-0.1, 5)
ax.set_ylim(-0.1, 5)
ax.grid()
ax.set_xlabel('Truth (y_test)')
ax.set_ylabel('Decision Tree prediction (y_predict)')
plt.show()

Let's try using `sklearn`s grid search tools.  This time we will try `RandomizedSearchCV`. When using `RandomizedSearchCV`, we will define a probability distribution (via scipy.stats library) for each feature we want to explore in our grid search. If a numpy array is provided, a value will be sampled uniformly from the array. 

In [ ]:
import scipy.stats as stats
hyperparameter_distributions = {
    'max_depth': stats.randint(1, 20),                       # defining a random integer probabl dist
    'min_samples_split': np.arange(5, 105, 10).astype(int),  # if array of values provided, a value will be sampled uniformly
    'min_samples_leaf': np.arange(5, 105, 10).astype(int)
}

In [ ]:
# import object needed from skelarn
from sklearn.model_selection import RandomizedSearchCV

# instantiate DT mode
dtree = DecisionTreeRegressor()

# instantiate RandomizedSearchCV
random_search = RandomizedSearchCV(
    dtree, 
    param_distributions=hyperparameter_distributions,
    n_iter=100     # number of samples considered from definited probability distributions
) 

# note I did not specify cv here; default value is 5
random_search.fit(X_train, y_train);

In [ ]:
# printing best hyperparameter combo found
print(random_search.best_params_)

# I can pull out best performing model with random_search.best_estimator_ attribute

In [ ]:
y_predict = random_search.best_estimator_.predict(X_test)
print(f'Decision Tree model with best performance from hyperparameter search = {mean_squared_error(y_test, y_predict):.4f}')
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.scatter(y_test, y_predict, alpha=0.2, color='black')
ax.set_aspect('equal')
ax.set_xlim(-0.1, 5)
ax.set_ylim(-0.1, 5)
ax.grid()
ax.set_xlabel('Truth (y_test)')
ax.set_ylabel('Decision Tree prediction (y_predict)');

In [ ]:
# we will create these plots later on in this notebook, so I will put the code into a function 
def plot_truth_v_prediction(ax,y_test,y_predict,ylabel='prediction (y_predict)'):
    ax.scatter(y_test, y_predict, alpha=0.2, color='black')
    ax.set_aspect('equal')
    ax.set_xlim(-0.1, 5)
    ax.set_ylim(-0.1, 5)
    ax.grid()
    ax.set_xlabel('Truth (y_test)')
    ax.set_ylabel(ylabel)

Ok, great we now have improved our performance compared to where we got to using linear regression yesterday.  Next let's see if Ensemble methods can help us even more.

# Random Forest 

Next we will build our random forest model.  We will gain perform a Grid Search using `RandomSearchCV`.  Note in this example we pass in KFold object to have a bit more control over the cross validation process. 

In [ ]:
model = RandomForestRegressor(random_state=0,  # seed for random number generator
                              n_jobs=-1)       # use all available cores to train models

param_grid = {"max_depth": np.arange(1, 20, 2).astype(int),  # defining a random integer probabl dist
    'min_samples_split': np.arange(5, 105, 10).astype(int),               # if array of values provided, a value will be sampled uniformly
    'min_samples_leaf': np.arange(5, 105, 10).astype(int)}

# We use K-Fold cross-validator.
# This give us more control over splits in cross validation
cv = KFold(n_splits=4, shuffle=True, random_state=0)


grid_search_rf = RandomizedSearchCV(
    estimator=model,
    param_distributions=hyperparameter_distributions,
    n_iter=100,
    return_train_score=True,      # returns data on training times and detailed analysis of evaluation metrics
    cv=cv,
    scoring='neg_mean_squared_error'
).fit(X_train, y_train)

In [ ]:
y_predict = grid_search_rf.best_estimator_.predict(X_test)
print(f'First Random Forest model performance MSE = {mean_squared_error(y_test, y_predict):.4f}')

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
plot_truth_v_prediction(ax,y_test,y_predict,ylabel='Random Forest prediction (y_predict)');

### Random Forest: Feature importance based on mean decrease in impurity (Gini Importance)

Tutorial is based on https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html.


Feature importances are provided by the fitted attribute feature_importances_ and they are computed as the mean and standard deviation of accumulation of the impurity decrease within each tree.

In [ ]:
importances = grid_search_rf.best_estimator_.feature_importances_
std = np.std([tree.feature_importances_ for tree in grid_search_rf.best_estimator_.estimators_], axis=0)

Let’s plot the impurity-based importance.

In [ ]:
forest_importances = pd.Series(grid_search_rf.best_estimator_.feature_importances_, index=features)
fig, ax = plt.subplots()
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

The most “important” features in this model are the F125W and F606W fluxes. Parameters such as id, ra, dec, x, and y understandably have very little influence on the predictions.

## 3. Gradient Boosting: Xgboost

In [ ]:
# instantiate and fit model 
model = xgb.XGBRegressor().fit(X_train, y_train)

# We evaluate a model and make predictions as we would do in scikit-learn.
# make predictions
predictions = model.predict(X_test)
print("Mean Squared Error : " + str(mean_squared_error(predictions, y_test)))

### Model Tuning

XGBoost has a few parameters that can dramatically affect your model's accuracy and training speed. In this tutorial we will run parameter scan on n_estimators and early_stopping_rounds.

N_estimators is the number of gradient boosted trees (Equivalent to number of boosting rounds).

Early_stopping_rounds offers a way to automatically find the ideal value. Early stopping causes the model to stop iterating when the validation score stops improving, even if we aren't at the hard stop for n_estimators.

Here we set a high value for n_estimators and then use early_stopping_rounds to find the optimal time to stop iterating.

In [ ]:
model = xgb.XGBRegressor(n_estimators=100, early_stopping_rounds=5).fit(
                X_train, y_train, 
                eval_set=[(X_test, y_test)], verbose=True);

In [ ]:
predictions = model.predict(X_test)
print("Mean Squared Error : " + str(mean_squared_error(predictions, y_test)))

# Exercise 

Can you get even better performance out of our model? 

A few ideas: 
1. Try a grid search again with Xgboost 
2. What happens if we remove or change features in our model? 
3. When if we change how we treat our missing values in the dataset?

Add your model results plus your best XGBoost model to the final performance.

## Compare final performance

Finally, lets compare our Random Forest and Gradient Boosting models with the results from Skelton et. al 2014. 

In [ ]:
#y_predict_xgb = # add your best estimator for xgboost here
y_predict_rf = grid_search_rf.best_estimator_.predict(X_validate)
y_skelton2014 = tab['z_peak'].values[indices_validate]

#print(f'Gradient Boosting model performance MSE = {mean_squared_error(y_validate, y_predict_xgb):.4f}')
print(f'Random Forest model performance MSE = {mean_squared_error(y_validate, y_predict_rf):.4f}')
print(f'Skelton 2014 et. al. model performance MSE = {mean_squared_error(y_validate, y_skelton2014):.4f}')

fig, ax = plt.subplots(1, 3, figsize=(12, 4))

plot_truth_v_prediction(ax[0],y_test,y_predict,ylabel='Random Forest prediction (y_predict)')
#plot_truth_v_prediction(ax[1],y_test,y_predict,ylabel='Gradient Boosting prediction (y_predict)')
plot_truth_v_prediction(ax[2],y_test,y_predict,ylabel='Skelton et al. 2014 model (z_peak)')
ax[0].set_title('Random Forest')
ax[1].set_title('XGB')
ax[2].set_title('skelton2014');